In [5]:
# 필수 라이브러리 일괄 임포트
import json
import os
import time
from datetime import datetime
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [6]:
def print_log(level, message):
    """시간이 포함된 규격화된 로그를 출력하는 함수"""
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{now}] [{level}] {message}")

# 1. 임베딩 모델 로드
print_log("INFO", "임베딩 모델 로드 시작 (GPU: cuda)")
embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
print_log("SUCCESS", "임베딩 모델 로드 완료")

# 모델 정상 작동 테스트
test_vector = embedding_model.embed_query("나이아신아마이드 EWG 등급")
print_log("INFO", f"테스트 임베딩 차원 수 확인: {len(test_vector)}")

[2026-04-22 10:52:00] [INFO] 임베딩 모델 로드 시작 (GPU: cuda)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[2026-04-22 10:52:22] [SUCCESS] 임베딩 모델 로드 완료
[2026-04-22 10:52:23] [INFO] 테스트 임베딩 차원 수 확인: 768


In [7]:
print_log("INFO", "FAISS 인덱스 일괄 구축 파이프라인 시작")
print("=" * 60)

for preset_id in range(1, 5):
    start_time = time.time()
    print_log("START", f"Preset {preset_id} 처리 시작")

    # 1. 파일 경로 설정 및 확인
    input_file = f"/content/ingredient_chunks_preset_{preset_id}.json"
    save_path = f"/content/drive/MyDrive/data/faiss_index_preset_{preset_id}"

    if not os.path.exists(input_file):
        print_log("ERROR", f"파일을 찾을 수 없습니다. 경로 스킵: {input_file}")
        print("-" * 60)
        continue

    # 2. JSON 데이터 로드
    with open(input_file, "r", encoding="utf-8") as f:
        all_chunks = json.load(f)
    chunk_count = len(all_chunks)
    print_log("INFO", f"JSON 로드 완료 (입력 청크 수: {chunk_count:,}개)")

    # 3. LangChain Document 객체 변환
    docs = [
        Document(
            page_content=chunk["page_content"],
            metadata=chunk["metadata"]
        )
        for chunk in all_chunks
    ]

    # 4. FAISS 인덱스 구축
    print_log("PROCESS", "FAISS 인덱스 구축 중... (GPU 연산)")
    vectorstore = FAISS.from_documents(
        documents=docs,
        embedding=embedding_model,
    )
    vector_count = vectorstore.index.ntotal
    print_log("INFO", f"FAISS 인덱스 구축 완료 (생성된 벡터 수: {vector_count:,}개)")

    # 데이터 유실 체크 (입력 청크 수와 출력 벡터 수가 다르면 경고)
    if chunk_count != vector_count:
        print_log("WARNING", f"데이터 불일치 발생! (청크: {chunk_count} vs 벡터: {vector_count})")

    # 5. 로컬(구글 드라이브) 저장
    vectorstore.save_local(save_path)

    # 6. 소요 시간 계산 및 최종 완료 로그
    elapsed_time = time.time() - start_time
    print_log("SUCCESS", f"Preset {preset_id} 저장 완료 (소요 시간: {elapsed_time:.2f}초)")
    print_log("PATH", f"저장 위치: {save_path}")
    print("-" * 60)

print_log("END", "모든 프리셋의 FAISS 인덱스 구축 및 저장이 완료되었습니다.")

[2026-04-22 10:52:29] [INFO] FAISS 인덱스 일괄 구축 파이프라인 시작
[2026-04-22 10:52:29] [START] Preset 1 처리 시작
[2026-04-22 10:52:29] [INFO] JSON 로드 완료 (입력 청크 수: 42,227개)
[2026-04-22 10:52:30] [PROCESS] FAISS 인덱스 구축 중... (GPU 연산)
[2026-04-22 10:56:14] [INFO] FAISS 인덱스 구축 완료 (생성된 벡터 수: 42,227개)
[2026-04-22 10:56:16] [SUCCESS] Preset 1 저장 완료 (소요 시간: 227.28초)
[2026-04-22 10:56:16] [PATH] 저장 위치: /content/drive/MyDrive/data/faiss_index_preset_1
------------------------------------------------------------
[2026-04-22 10:56:16] [START] Preset 2 처리 시작
[2026-04-22 10:56:16] [INFO] JSON 로드 완료 (입력 청크 수: 42,227개)
[2026-04-22 10:56:17] [PROCESS] FAISS 인덱스 구축 중... (GPU 연산)
[2026-04-22 11:00:21] [INFO] FAISS 인덱스 구축 완료 (생성된 벡터 수: 42,227개)
[2026-04-22 11:00:22] [SUCCESS] Preset 2 저장 완료 (소요 시간: 246.46초)
[2026-04-22 11:00:22] [PATH] 저장 위치: /content/drive/MyDrive/data/faiss_index_preset_2
------------------------------------------------------------
[2026-04-22 11:00:22] [START] Preset 3 처리 시작
[2026-04-22 11:00:23] [IN